# Задание 2 Вопросы к базе знаний

Контрольная работа по дисциплине «Семантические технологии». Предметная область — 52 библиотеки экосистемы PyTorch. Эталонные ответы относятся к графу knowledge_graph.ttl и корпусу data/corpus.jsonl из текущего снимка WIKIBASE.

[Условие контрольной работы](https://docs.google.com/document/d/1x7V7MV4sEXqoKcddzE-0TX4NfRwtbbWqO9DLcnxSyi8/edit?usp=sharing)

In [1]:
import json
from pathlib import Path

from rdflib import Graph

project = Path("..")
graph = Graph().parse(project / "knowledge_graph.ttl", format="turtle")
corpus = {}
with (project / "data" / "corpus.jsonl").open(encoding="utf-8") as file:
    for line in file:
        document = json.loads(line)
        corpus[document["id"]] = document


def run(query):
    result = graph.query(query)
    rows = list(result)
    print(*result.vars, sep=" | ")
    for row in rows:
        print(*row, sep=" | ")
    return rows


def check_quote(qid, quote):
    document = corpus[qid]
    assert quote in " ".join(document["text"].split())
    print(document["name"], qid)
    print(quote)

## Вопросы к графу знаний

### 1. У каких библиотек экосистемы PyTorch в графе зарегистрировано более 20 версий? Укажите названия и число версий.

**Тип.** Графовый — фильтр по числу

**Эталонный ответ.** В графе 23 такие записи.

| Библиотека | Число версий |
| --- | --- |
| pytorch-pretrained-bert | 242 |
| pytorch-transformers | 242 |
| PyTorch Lightning | 208 |
| accelerate | 67 |
| torchmetrics | 63 |
| open-clip-torch | 57 |
| Pyserini | 57 |
| botorch | 55 |
| rotary-embedding-torch | 53 |
| timm | 51 |
| allennlp | 47 |
| kornia | 42 |
| catalyst | 36 |
| peft | 36 |
| pyro-ppl | 36 |
| torchvision | 29 |
| gpytorch | 28 |
| flair | 27 |
| nerfacc | 26 |
| torchaudio | 24 |
| hummingbird-ml | 23 |
| PyKEEN | 22 |
| triton | 22 |

In [2]:
run("""
PREFIX pte: <http://example.org/pytorch-ecosystem#>
SELECT ?name ?count WHERE {
  ?library pte:libraryName ?name ; pte:versionCount ?count .
  FILTER(?count > 20)
}
ORDER BY DESC(?count) LCASE(STR(?name)) ?name
""");

name | count
pytorch-pretrained-bert | 242
pytorch-transformers | 242
PyTorch Lightning | 208
accelerate | 67
torchmetrics | 63
open-clip-torch | 57
Pyserini | 57
botorch | 55
rotary-embedding-torch | 53
timm | 51
allennlp | 47
kornia | 42
catalyst | 36
peft | 36
pyro-ppl | 36
torchvision | 29
gpytorch | 28
flair | 27
nerfacc | 26
torchaudio | 24
hummingbird-ml | 23
PyKEEN | 22
triton | 22


### 2. Сколько различных библиотек связано в графе с каждой лицензией?

**Тип.** Графовый — агрегация

**Эталонный ответ.**

| Лицензия | Число библиотек |
| --- | --- |
| лицензия MIT | 23 |
| Apache License 2.0 | 21 |
| модифицированная лицензия BSD | 6 |
| 2-пунктная лицензия BSD | 2 |
| Free To Use But Restricted | 1 |

В подсчёт также входит центральный узел PyTorch.

In [3]:
run("""
PREFIX pte: <http://example.org/pytorch-ecosystem#>
SELECT ?licenseName (COUNT(DISTINCT ?library) AS ?count) WHERE {
  ?library pte:hasLicense ?license .
  ?license pte:licenseName ?licenseName .
}
GROUP BY ?licenseName
ORDER BY DESC(?count) ?licenseName
""");

licenseName | count
лицензия MIT | 23
Apache License 2.0 | 21
модифицированная лицензия BSD | 6
2-пунктная лицензия BSD | 2
Free To Use But Restricted | 1


### 3. Какие пять библиотек занимают первые места по числу зарегистрированных в графе версий? При равенстве расположите их по названию.

**Тип.** Графовый — сортировка

**Эталонный ответ.**

| Библиотека | Число версий |
| --- | --- |
| pytorch-pretrained-bert | 242 |
| pytorch-transformers | 242 |
| PyTorch Lightning | 208 |
| accelerate | 67 |
| torchmetrics | 63 |

In [4]:
run("""
PREFIX pte: <http://example.org/pytorch-ecosystem#>
SELECT ?name ?count WHERE {
  ?library pte:libraryName ?name ; pte:versionCount ?count .
}
ORDER BY DESC(?count) LCASE(STR(?name)) ?name
LIMIT 5
""");

name | count
pytorch-pretrained-bert | 242
pytorch-transformers | 242
PyTorch Lightning | 208
accelerate | 67
torchmetrics | 63


### 4. Какие библиотеки компьютерного зрения зависят от PyTorch и распространяются по лицензии Apache License 2.0? Укажите ссылки на их репозитории.

**Тип.** Графовый — многошаговый переход

**Эталонный ответ.**

| Библиотека | Репозиторий |
| --- | --- |
| effdet | https://github.com/rwightman/efficientdet-pytorch |
| gfpgan | https://github.com/TencentARC/GFPGAN |
| kornia | https://github.com/kornia/kornia |
| timm | https://github.com/huggingface/pytorch-image-models |

Адрес получен переходом по двум рёбрам `hasRepository` и `repositoryURL`.

In [5]:
run("""
PREFIX pte: <http://example.org/pytorch-ecosystem#>
SELECT DISTINCT ?name ?repoUrl WHERE {
  ?library a pte:ComputerVisionLibrary ;
           pte:libraryName ?name ;
           pte:dependsOnSoftware pte:PyTorch ;
           pte:hasLicense pte:License_Apache_License_2_0 ;
           pte:hasRepository/pte:repositoryURL ?repoUrl .
}
ORDER BY ?name ?repoUrl
""");

name | repoUrl
effdet | https://github.com/rwightman/efficientdet-pytorch
gfpgan | https://github.com/TencentARC/GFPGAN
kornia | https://github.com/kornia/kornia
timm | https://github.com/huggingface/pytorch-image-models


### 5. Какие десять различных сочетаний библиотеки, номера версии и даты выпуска идут первыми по дате публикации в 2023 году? При равенстве дат отсортируйте по названию библиотеки и номеру версии как строке.

**Тип.** Графовый — временной квалификатор

**Эталонный ответ.**

| Библиотека | Версия | Дата (UTC) |
| --- | --- | --- |
| pyro-ppl | 1.8.4 | 2023-01-03 |
| gpytorch | 1.9.1 | 2023-01-04 |
| open-clip-torch | 2.9.2 | 2023-01-05 |
| botorch | 0.8.1 | 2023-01-06 |
| open-clip-torch | 2.9.3 | 2023-01-09 |
| PyTorch Lightning | 1.9.0 | 2023-01-17 |
| nerfacc | 0.3.3 | 2023-01-18 |
| open-clip-torch | 2.10.0 | 2023-01-23 |
| open-clip-torch | 2.10.1 | 2023-01-24 |
| pytorch-pretrained-bert | 4.26.0 | 2023-01-25 |

`releaseDate` хранит время публикации конкретной версии и соответствует квалификатору `pq:P577` исходного утверждения Wikidata о версии `p:P348`.

In [6]:
run("""
PREFIX pte: <http://example.org/pytorch-ecosystem#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT DISTINCT ?name ?version ?date WHERE {
  ?library pte:libraryName ?name ; pte:hasVersion ?release .
  ?release pte:versionIdentifier ?version ; pte:releaseDate ?date .
  FILTER(?date >= "2023-01-01T00:00:00Z"^^xsd:dateTime &&
         ?date < "2024-01-01T00:00:00Z"^^xsd:dateTime)
}
ORDER BY ?date ?name ?version
LIMIT 10
""");

name | version | date
pyro-ppl | 1.8.4 | 2023-01-03T00:00:00+00:00
gpytorch | 1.9.1 | 2023-01-04T00:00:00+00:00
open-clip-torch | 2.9.2 | 2023-01-05T00:00:00+00:00
botorch | 0.8.1 | 2023-01-06T00:00:00+00:00
open-clip-torch | 2.9.3 | 2023-01-09T00:00:00+00:00
PyTorch Lightning | 1.9.0 | 2023-01-17T00:00:00+00:00
nerfacc | 0.3.3 | 2023-01-18T00:00:00+00:00
open-clip-torch | 2.10.0 | 2023-01-23T00:00:00+00:00
open-clip-torch | 2.10.1 | 2023-01-24T00:00:00+00:00
pytorch-pretrained-bert | 4.26.0 | 2023-01-25T00:00:00+00:00


### 6. Какие записи библиотек классифицированы в графе как библиотеки обработки естественного языка (NLP)? Укажите адреса их репозиториев.

**Тип.** Графовый — классификация по классу

**Эталонный ответ.** В графе 12 записей.

| Библиотека | Репозиторий |
| --- | --- |
| STonKGs | https://github.com/stonkgs/stonkgs |
| Stanza | https://github.com/stanfordnlp/stanza |
| allennlp | https://github.com/allenai/allennlp |
| attacut | https://github.com/PyThaiNLP/attacut |
| bert-score | https://github.com/Tiiiger/bert_score |
| flair | https://github.com/flairNLP/flair |
| fschat | https://github.com/lm-sys/fastchat |
| peft | https://github.com/huggingface/peft |
| pytorch-pretrained-bert | https://github.com/huggingface/transformers |
| pytorch-transformers | https://github.com/huggingface/transformers |
| snorkel | https://github.com/snorkel-team/snorkel |
| torchtext | https://github.com/pytorch/text |

`pytorch-pretrained-bert` и `pytorch-transformers` имеют разные идентификаторы Wikidata и один URL репозитория.

In [7]:
run("""
PREFIX pte: <http://example.org/pytorch-ecosystem#>
SELECT DISTINCT ?name ?repoUrl WHERE {
  ?library a pte:NaturalLanguageProcessingLibrary ;
           pte:libraryName ?name ;
           pte:hasRepository/pte:repositoryURL ?repoUrl .
}
ORDER BY ?name ?repoUrl
""");

name | repoUrl
STonKGs | https://github.com/stonkgs/stonkgs
Stanza | https://github.com/stanfordnlp/stanza
allennlp | https://github.com/allenai/allennlp
attacut | https://github.com/PyThaiNLP/attacut
bert-score | https://github.com/Tiiiger/bert_score
flair | https://github.com/flairNLP/flair
fschat | https://github.com/lm-sys/fastchat
peft | https://github.com/huggingface/peft
pytorch-pretrained-bert | https://github.com/huggingface/transformers
pytorch-transformers | https://github.com/huggingface/transformers
snorkel | https://github.com/snorkel-team/snorkel
torchtext | https://github.com/pytorch/text


## Вопросы к текстовому корпусу

### 7. Какие виды алгоритмов предоставляет Kornia, согласно вступлению её README?

**Тип.** Текстовый, назначение библиотеки

**Эталонный ответ.** Kornia предоставляет дифференцируемые алгоритмы обработки изображений и геометрического компьютерного зрения.

**Цитата.** «a rich set of differentiable image processing and geometric vision algorithms»

**Почему нужен текст.** В графе указаны класс Kornia и общее описание. Два направления её алгоритмов, обработка изображений и геометрическое зрение, явно не перечислены.

[README в репозитории автора](https://github.com/kornia/kornia/blob/5bc80bd1918f03dc55bac306004ad0b39b892050/README.md#L34-L35)

In [8]:
check_quote('Q107382312', 'a rich set of differentiable image processing and geometric vision algorithms')

kornia Q107382312
a rich set of differentiable image processing and geometric vision algorithms


### 8. Где началась разработка Pyro и кто поддерживает проект, согласно README?

**Тип.** Текстовый, история проекта

**Эталонный ответ.** Pyro начали разрабатывать в Uber AI. Проект поддерживают участники сообщества.

**Цитата.** «Pyro was originally developed at Uber AI and is now actively maintained by community contributors»

**Почему нужен текст.** Граф описывает классификацию, лицензии, репозитории, зависимости и версии. Организации, создавшие и сопровождающие проект, в этих данных не указаны.

[README в репозитории автора](https://github.com/pyro-ppl/pyro/blob/a31bc1784707268cd79df77ead4c7a47062f0193/README.md#L31)

In [9]:
check_quote('Q107382321', 'Pyro was originally developed at Uber AI and is now actively maintained by community contributors')

pyro-ppl Q107382321
Pyro was originally developed at Uber AI and is now actively maintained by community contributors


### 9. На каких вычислительных конфигурациях Accelerate позволяет запускать один скрипт PyTorch и какие форматы смешанной точности перечислены в README?

**Тип.** Текстовый, возможности запуска

**Эталонный ответ.** README перечисляет один CPU, один GPU, несколько GPU и TPU. Для смешанной точности указаны FP8, FP16 и BF16.

**Цитата.** «single or distributed node setting (single CPU, single GPU, multi-GPUs and TPUs) as well as with or without mixed precision (fp8, fp16, bf16)»

**Почему нужен текст.** В графе отсутствуют сведения о конфигурациях вычислительных устройств и форматах смешанной точности.

[README в репозитории автора](https://github.com/huggingface/accelerate/blob/f47439c1fb24721ea71ab98f9b512726b5a97828/README.md#L84)

In [10]:
check_quote('Q116908792', 'single or distributed node setting (single CPU, single GPU, multi-GPUs and TPUs) as well as with or without mixed precision (fp8, fp16, bf16)')

accelerate Q116908792
single or distributed node setting (single CPU, single GPU, multi-GPUs and TPUs) as well as with or without mixed precision (fp8, fp16, bf16)


### 10. Какие методы атрибуции перечислены во вступительном абзаце README библиотеки Captum?

**Тип.** Текстовый, методы интерпретации

**Эталонный ответ.** Во вступительном абзаце перечислены интегрированные градиенты (Integrated Gradients), карты значимости (saliency maps), SmoothGrad и VarGrad.

**Цитата.** «integrated gradients, saliency maps, smoothgrad, vargrad»

**Почему нужен текст.** В графе для Captum зафиксированы класс и метаданные. Методы атрибуции и их названия в нём не перечислены.

[README в репозитории автора](https://github.com/meta-pytorch/captum/blob/b89cccbe0add104f734793be69488c078164c532/README.md#L23-L26)

In [11]:
check_quote('Q107382311', 'integrated gradients, saliency maps, smoothgrad, vargrad')

captum Q107382311
integrated gradients, saliency maps, smoothgrad, vargrad


## Комбинированные вопросы

### 11. Какая библиотека компьютерного зрения с лицензией Apache 2.0 имеет больше всего зарегистрированных версий в графе? Приведите четыре семейства моделей с предобученными вариантами из её README.

**Тип.** Комбинированный — граф и текст

**Часть из графа.** Граф выбирает **timm** (`Q107382194`) с **51 версией** среди библиотек компьютерного зрения под лицензией Apache License 2.0.

**Часть из текста.** В разделе Models полного README перечислены архитектуры и указано, что каждое семейство включает варианты с предобученными весами. Для ответа используется полный README по ссылке, поскольку локальный корпус содержит только первые 4000 символов этого файла.

**Эталонный ответ.** timm (51 зарегистрированная версия). Примеры семейств с предобученными вариантами — ConvNeXt, ResNet, Swin Transformer и Vision Transformer (ViT).

**Цитата.** «All model architecture families include variants with pretrained weights.»

[README в репозитории автора](https://github.com/huggingface/pytorch-image-models/blob/43b549bfe9d680bc94284cfd0b611152d18e97d7/README.md#L239-L376)

In [12]:
rows = run("""
PREFIX pte: <http://example.org/pytorch-ecosystem#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
SELECT ?name ?count ?qid WHERE {
  ?library a pte:ComputerVisionLibrary ;
           pte:libraryName ?name ;
           pte:hasLicense pte:License_Apache_License_2_0 ;
           pte:versionCount ?count ;
           owl:sameAs ?wikidata .
  BIND(STRAFTER(STR(?wikidata), "/entity/") AS ?qid)
}
ORDER BY DESC(?count) ?name ?qid
LIMIT 1
""")
assert len(rows) == 1 and str(rows[0].qid) == 'Q107382194'
print(corpus[str(rows[0].qid)]["source_url"])

name | count | qid
timm | 51 | Q107382194
https://raw.githubusercontent.com/huggingface/pytorch-image-models/master/README.md


### 12. Какая библиотека вероятностного и графового машинного обучения с лицензией MIT имеет ровно 22 зарегистрированные версии в графе? Какие операции над моделями эмбеддингов указаны в её README и какая модель показана в примере?

**Тип.** Комбинированный — граф и текст

**Часть из графа.** Граф выбирает **PyKEEN** (`Q96631404`), библиотеку класса `ProbabilisticAndGraphLibrary` под лицензией MIT с **22 версиями**.

**Часть из текста.** В README сформулировано назначение пакета и показан пример обучения и оценки TransE на наборе Nations.

**Эталонный ответ.** PyKEEN (лицензия MIT, 22 зарегистрированные версии) предназначена для обучения и оценки моделей эмбеддингов графов знаний, в том числе мультимодальных. В примере обучают и оценивают TransE на наборе Nations.

**Цитата.** «train and evaluate knowledge graph embedding models (incorporating multi-modal information).»

[README в репозитории автора](https://github.com/pykeen/pykeen/blob/5beaada8623a7bc368db9f29a7a43ddd237888a6/README.md#L41-L91)

In [13]:
rows = run("""
PREFIX pte: <http://example.org/pytorch-ecosystem#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
SELECT ?name ?count ?qid WHERE {
  ?library a pte:ProbabilisticAndGraphLibrary ;
           pte:libraryName ?name ;
           pte:hasLicense pte:License_лицензия_MIT ;
           pte:versionCount ?count ;
           owl:sameAs ?wikidata .
  BIND(STRAFTER(STR(?wikidata), "/entity/") AS ?qid)
  FILTER(?count = 22)
}
ORDER BY ?name ?wikidata
""")
assert len(rows) == 1 and str(rows[0].qid) == 'Q96631404'
check_quote(str(rows[0].qid), 'train and evaluate knowledge graph embedding models (incorporating multi-modal information).')

name | count | qid
PyKEEN | 22 | Q96631404
PyKEEN Q96631404
train and evaluate knowledge graph embedding models (incorporating multi-modal information).
